In [ ]:
import sys, os, subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run(["pip", "install", "-q", "mplsoccer", "scipy", "pillow"], check=True)
    REPO_URL = "https://github.com/antiachismosocialclub/worldcup.git"
    if not os.path.exists("/content/worldcup"):
        subprocess.run(["git", "clone", REPO_URL, "/content/worldcup"], check=True)
    os.chdir("/content/worldcup/notebooks")
    print("Colab: repo clonado e diretorio configurado.")
else:
    # local: garante que estamos dentro de notebooks/
    nb_dir = os.path.join(os.path.dirname(os.path.abspath("__file__")), "notebooks")
    if os.path.basename(os.getcwd()) != "notebooks" and os.path.exists(nb_dir):
        os.chdir(nb_dir)
    print("Local OK:", os.getcwd())


# Quem substitui Raphinha na Copa do Mundo 2026?
**Fonte:** SofaScore - coleta manual via extensao Chrome (22/06/2026)
**Autor:** Analise colaborativa baseada em dados publicos
**Data:** 22 de junho de 2026

---

Raphinha sofreu lesao muscular (hamstring) durante Brasil x Haiti e esta fora dos proximos jogos.
Tres candidatos a ponta direita: **Luiz Henrique** (Zenit/RFPL), **Rayan** (Bournemouth/PL), **Endrick** (Lyon/Ligue 1).

**Novidade desta versao:** o `ajuste_tatico` deixou de ser subjetivo.
Foi substituido por similaridade coseno entre os mapas de calor do SofaScore de cada candidato
e o mapa de calor do Raphinha - calculada em dois cenarios:

| Cenario | Referencia | Logica |
|---|---|---|
| **A** | Raphinha La Liga (22 jogos) | Quem joga mais parecido com o Raphinha do Barca/Flick? |
| **B** | Raphinha Copa (2 jogos) | Quem joga mais parecido com como Ancelotti esta usando o Raphinha? |


## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import json
from math import pi
from pathlib import Path
from PIL import Image as PILImage
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from mplsoccer import Pitch

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# ── Paleta de cores ──────────────────────────────────────────────────────────
COLORS = {
    "Raphinha":      "#1f77b4",
    "Luiz Henrique": "#ff7f0e",
    "Rayan":         "#2ca02c",
    "Endrick":       "#d62728",
}
PLAYER_COLORS = {
    "Raphinha (La Liga)": COLORS["Raphinha"],
    "Raphinha (Copa)":    "#5ba4cf",
    "Luiz Henrique":      COLORS["Luiz Henrique"],
    "Rayan":              COLORS["Rayan"],
    "Endrick":            COLORS["Endrick"],
}
CANDIDATES = ["Luiz Henrique", "Rayan", "Endrick"]
ALL_PLAYERS = ["Raphinha", "Luiz Henrique", "Rayan", "Endrick"]

# ── Informacoes dos jogadores ─────────────────────────────────────────────────
IMG_BASE = Path("../data/raw/images")

players_info = [
    {
        "name": "Raphinha", "role": "REFERENCIA",
        "team": "FC Barcelona", "league": "La Liga",
        "color": COLORS["Raphinha"],
        "player_img": str(IMG_BASE / "player_raphinha.png"),
        "team_img":   str(IMG_BASE / "team_barcelona.png"),
        "league_img": str(IMG_BASE / "league_laliga.png"),
    },
    {
        "name": "Luiz Henrique", "role": "CANDIDATO",
        "team": "Zenit", "league": "RFPL",
        "color": COLORS["Luiz Henrique"],
        "player_img": str(IMG_BASE / "player_luiz_henrique.png"),
        "team_img":   str(IMG_BASE / "team_zenit.png"),
        "league_img": str(IMG_BASE / "league_rfpl.png"),
    },
    {
        "name": "Rayan", "role": "CANDIDATO",
        "team": "Bournemouth", "league": "Premier League",
        "color": COLORS["Rayan"],
        "player_img": str(IMG_BASE / "player_rayan.png"),
        "team_img":   str(IMG_BASE / "team_bournemouth.png"),
        "league_img": str(IMG_BASE / "league_premier.png"),
    },
    {
        "name": "Endrick", "role": "CANDIDATO",
        "team": "Olympique Lyonnais", "league": "Ligue 1",
        "color": COLORS["Endrick"],
        "player_img": str(IMG_BASE / "player_endrick.png"),
        "team_img":   str(IMG_BASE / "team_lyon.png"),
        "league_img": str(IMG_BASE / "league_ligue1.png"),
    },
]

PLAYER_PHOTO = {p["name"]: p["player_img"] for p in players_info}

# foto mapeada para labels de heatmap (Raphinha aparece duas vezes)
HEATMAP_PHOTO = {
    "Raphinha (La Liga)": PLAYER_PHOTO["Raphinha"],
    "Raphinha (Copa)":    PLAYER_PHOTO["Raphinha"],
    **{p: PLAYER_PHOTO[p] for p in CANDIDATES},
}

# ── Utilitarios de imagem ─────────────────────────────────────────────────────
def load_img(path, size=None):
    try:
        img = PILImage.open(path).convert("RGBA")
        if size:
            img = img.resize(size, PILImage.LANCZOS)
        return np.array(img)
    except:
        return None

def add_player_img(ax, img_path, zoom=0.18, xy=(0.5, 1.13)):
    img = load_img(img_path)
    if img is None:
        return
    im = OffsetImage(img, zoom=zoom)
    ab = AnnotationBbox(im, xy, xycoords="axes fraction", frameon=False, zorder=10)
    ax.add_artist(ab)

print("Setup OK")


## 2. Dataset

In [ ]:
df_all = pd.read_csv("../data/dataset_substituto_raphinha.csv")
df = df_all[df_all["player"].isin(CANDIDATES)].copy().reset_index(drop=True)

cols_show  = ["league", "matches", "minutes", "goals", "assists", "xg", "g_a_90", "shots_on_90"]
col_labels = ["Liga", "Jogos", "Min", "Gols", "Assist", "xG", "G+A/90", "Chutes/90"]
order      = ["Raphinha", "Luiz Henrique", "Rayan", "Endrick"]
rows       = df_all[df_all["player"].isin(order)].set_index("player").loc[order]

n_rows = len(rows)
n_cols = len(cols_show)
IMG_COL_W = 1.0
COL_W     = 1.1

fig_w = IMG_COL_W + n_cols * COL_W + 0.4
fig, ax = plt.subplots(figsize=(fig_w, n_rows * 1.05 + 0.7))
ax.set_xlim(0, fig_w)
ax.set_ylim(0, n_rows + 0.7)
ax.axis("off")
fig.patch.set_facecolor("#0d1117")

# header
header_cols = ["Jogador"] + col_labels
x_positions = [IMG_COL_W / 2] + [IMG_COL_W + COL_W * (i + 0.5) for i in range(n_cols)]
for xp, lbl in zip(x_positions, header_cols):
    ax.text(xp, n_rows + 0.3, lbl, ha="center", va="center",
            fontsize=8, fontweight="bold", color="#aaaaaa",
            transform=ax.transData)

# separator line
ax.axhline(n_rows, color="#333333", linewidth=0.8, xmin=0, xmax=1)

for row_i, player in enumerate(order):
    y = n_rows - row_i - 1
    row_color = "#161b22" if row_i % 2 == 0 else "#0d1117"
    ax.add_patch(mpatches.FancyBboxPatch(
        (0.05, y + 0.05), fig_w - 0.1, 0.88,
        boxstyle="round,pad=0.02", facecolor=row_color,
        edgecolor="none", zorder=0
    ))

    # foto
    try:
        img = np.array(PILImage.open(PLAYER_PHOTO[player]).convert("RGBA"))
        im = OffsetImage(img, zoom=0.30)
        ab = AnnotationBbox(im, (IMG_COL_W / 2, y + 0.5),
                            xycoords="data", frameon=False, zorder=5)
        ax.add_artist(ab)
    except:
        ax.text(IMG_COL_W / 2, y + 0.5, player[0], ha="center", va="center",
                fontsize=14, color=COLORS.get(player, "white"))

    # nome abaixo da foto
    tag = "(ref)" if player == "Raphinha" else ""
    ax.text(IMG_COL_W / 2, y + 0.08, f"{player} {tag}".strip(),
            ha="center", va="bottom", fontsize=6.5,
            color=COLORS.get(player, "white"), fontweight="bold")

    # valores das colunas
    for col_i, col in enumerate(cols_show):
        xp = IMG_COL_W + COL_W * (col_i + 0.5)
        val = rows.loc[player, col]
        if col == "xg":
            txt = "N/D" if pd.isna(val) else f"{val:.2f}"
        elif col in ("g_a_90", "shots_on_90"):
            txt = f"{val:.2f}" if not pd.isna(val) else "-"
        elif col == "minutes":
            txt = f"{int(val)}"
        elif col in ("goals", "assists", "matches"):
            txt = str(int(val)) if not pd.isna(val) else "-"
        else:
            txt = str(val)
        ax.text(xp, y + 0.5, txt, ha="center", va="center",
                fontsize=8.5, color="white")

ax.axhline(0, color="#333333", linewidth=0.5)
plt.title("Dataset — Temporada 2025/26 (SofaScore)",
          fontsize=12, fontweight="bold", color="white", pad=10)
plt.tight_layout()
plt.show()
print("Nota: xG indisponivel para Russian Premier League (RFPL — sem parceria Opta/StatsBomb).")


## 3. Perfis dos Jogadores

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 7))
fig.patch.set_facecolor("#f5f5f5")

for ax, info in zip(axes, players_info):
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.set_facecolor("#f5f5f5")

    ax.add_patch(mpatches.FancyBboxPatch(
        (0.03, 0.02), 0.94, 0.96, boxstyle="round,pad=0.02",
        facecolor="white", edgecolor=info["color"], linewidth=3, zorder=1))

    ax.add_patch(mpatches.FancyBboxPatch(
        (0.03, 0.73), 0.94, 0.25, boxstyle="round,pad=0.01",
        facecolor=info["color"], edgecolor="none", zorder=2))

    ax.text(0.5, 0.95, info["role"], ha="center", va="center",
            fontsize=7, fontweight="bold", color="white",
            transform=ax.transAxes, zorder=3)

    img = load_img(info["player_img"])
    if img is not None:
        im = OffsetImage(img, zoom=0.32)
        ab = AnnotationBbox(im, (0.5, 0.63), xycoords="axes fraction",
                            frameon=False, zorder=4)
        ax.add_artist(ab)

    ax.text(0.5, 0.48, info["name"], ha="center", va="center",
            fontsize=13, fontweight="bold", color="#1a1a1a",
            transform=ax.transAxes, zorder=3)
    ax.text(0.5, 0.41, info["team"], ha="center", va="center",
            fontsize=9, color="#555555", transform=ax.transAxes, zorder=3)
    ax.text(0.5, 0.35, info["league"], ha="center", va="center",
            fontsize=8, color="#888888", style="italic",
            transform=ax.transAxes, zorder=3)

    for img_path, xpos in [(info["team_img"], 0.22), (info["league_img"], 0.78)]:
        logo = load_img(img_path, size=(60, 60))
        if logo is not None:
            im_l = OffsetImage(logo, zoom=0.6)
            ab_l = AnnotationBbox(im_l, (xpos, 0.26), xycoords="axes fraction",
                                  frameon=False, zorder=4)
            ax.add_artist(ab_l)

    ax.plot([0.08, 0.92], [0.18, 0.18], color="#eeeeee", linewidth=1.2,
            transform=ax.transAxes, zorder=3)

    row = df_all[df_all["player"] == info["name"]].iloc[0]
    for i, (label, col) in enumerate([("G/90", "goals_90"), ("A/90", "assists_90"), ("G+A/90", "g_a_90")]):
        x = 0.18 + i * 0.32
        ax.text(x, 0.13, f"{row[col]:.2f}", ha="center", va="center",
                fontsize=12, fontweight="bold", color=info["color"],
                transform=ax.transAxes, zorder=3)
        ax.text(x, 0.07, label, ha="center", va="center",
                fontsize=7, color="#aaaaaa", transform=ax.transAxes, zorder=3)

plt.suptitle("Perfis dos Jogadores - Temporada 2025/26", fontsize=15,
             fontweight="bold", y=1.01, color="#1a1a1a")
plt.tight_layout(pad=1.5)
plt.show()


## 4. Desempenho Ofensivo (candidatos)

Comparacao das metricas por 90 minutos. Raphinha e o benchmark - linha tracejada nos graficos.

In [ ]:
raph = df_all[df_all["player"] == "Raphinha"].iloc[0]

metricas = [
    ("goals_90",      "Gols / 90 min"),
    ("assists_90",    "Assistencias / 90 min"),
    ("g_a_90",        "G+A / 90 min"),
    ("shots_90",      "Finalizacoes / 90 min"),
    ("shots_on_90",   "Fin. no Gol / 90 min"),
    ("key_passes_90", "Passes-Chave / 90 min"),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()

for idx, (col, title) in enumerate(metricas):
    ax = axes[idx]

    # descarta NaN (ex: key_passes_90 ausente para LH)
    df_plot = df[["player", col]].dropna(subset=[col]).reset_index(drop=True)
    palette  = [COLORS[p] for p in df_plot["player"]]

    bars = ax.bar(df_plot["player"], df_plot[col],
                  color=palette, alpha=0.85, edgecolor="white", linewidth=1.2)

    # benchmark Raphinha
    ref_val = raph[col]
    if pd.notna(ref_val):
        ax.axhline(ref_val, color=COLORS["Raphinha"], linestyle="--", linewidth=1.5, alpha=0.7)
        ax.text(len(df_plot) - 0.45, ref_val * 1.02,
                f"Raphinha\n{ref_val:.2f}", color=COLORS["Raphinha"],
                fontsize=8, va="bottom", ha="right")

    # valor acima de cada barra
    col_max = df_plot[col].max()
    for bar, val in zip(bars, df_plot[col]):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + col_max * 0.015,
                f"{val:.2f}", ha="center", fontsize=9, fontweight="bold")

    # remove labels de texto do eixo x e adiciona fotos
    ax.tick_params(axis="x", labelbottom=False, length=0)
    for i, player in enumerate(df_plot["player"]):
        img = load_img(PLAYER_PHOTO[player])
        if img is not None:
            im = OffsetImage(img, zoom=0.17)
            ab = AnnotationBbox(im, (i, -0.13),
                               xycoords=("data", "axes fraction"),
                               frameon=False, clip_on=False, zorder=10)
            ax.add_artist(ab)

    ax.set_title(title, fontweight="bold")
    ax.set_ylabel("")
    ax.set_xlabel("")

plt.suptitle("Metricas Ofensivas por 90 min - Temporada 2025/26",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.subplots_adjust(bottom=0.10)
plt.show()


## 5. Analise Posicional - Mapas de Calor no Campo

Coordenadas extraidas via SofaScore (extensao Chrome).
- **x**: 0 = gol proprio, 100 = gol adversario
- **y**: 0 = flanco direito, 100 = flanco esquerdo
- Artefato filtrado: `x>=98, y<=1` (canto de escanteio)


In [ ]:
from matplotlib.patches import Arc, Circle, Rectangle
from scipy.ndimage import gaussian_filter

# ── Proporcoes FIFA 105x68m normalizadas para coordenadas SofaScore 0-100 ────
_F = dict(
    pen_x   = 15.71, pen_y0  = 20.4,  pen_h  = 59.2,
    gbox_x  = 5.24,  gbox_y0 = 36.5,  gbox_h = 27.0,
    goal_y0 = 44.6,  goal_h  = 10.8,
    pspot   = 10.5,  arc_d   = 18.3,  cc_r   = 9.16,
)

def draw_field(ax, bg="#1a472a", lc="white", lw=1.2):
    # Campo mais largo que alto (proporcao ~1.5:1 como campo real)
    ax.set_xlim(-2, 102)
    ax.set_ylim(-3, 103)   # eixo y levemente maior para compensar aspecto
    ax.set_facecolor(bg)
    ax.axis("off")
    kw = dict(color=lc, lw=lw, zorder=4)
    p  = dict(fill=False, edgecolor=lc, lw=lw, zorder=4)

    ax.add_patch(Rectangle((0, 0), 100, 100, **p))
    ax.plot([50, 50], [0, 100], **kw)
    ax.add_patch(Circle((50, 50), _F["cc_r"], **p))
    ax.plot(50, 50, "o", color=lc, ms=2, zorder=4)

    for x0, dx in [(0, _F["pen_x"]), (100-_F["pen_x"], _F["pen_x"])]:
        ax.add_patch(Rectangle((x0, _F["pen_y0"]), dx, _F["pen_h"], **p))
    for x0, dx in [(0, _F["gbox_x"]), (100-_F["gbox_x"], _F["gbox_x"])]:
        ax.add_patch(Rectangle((x0, _F["gbox_y0"]), dx, _F["gbox_h"], **p))
    for x0, dx in [(-2, 2), (100, 2)]:
        ax.add_patch(Rectangle((x0, _F["goal_y0"]), dx, _F["goal_h"],
                               facecolor="#555", edgecolor=lc, lw=lw, zorder=4))
    for px in [_F["pspot"], 100-_F["pspot"]]:
        ax.plot(px, 50, "o", color=lc, ms=2, zorder=4)
    ax.add_patch(Arc((_F["pspot"],      50), _F["arc_d"], _F["arc_d"], angle=0, theta1=308, theta2=52,  **kw))
    ax.add_patch(Arc((100-_F["pspot"], 50), _F["arc_d"], _F["arc_d"], angle=0, theta1=128, theta2=232, **kw))
    for cx, cy, t1, t2 in [(0,0,0,90),(100,0,90,180),(100,100,180,270),(0,100,270,360)]:
        ax.add_patch(Arc((cx, cy), 2.5, 2.5, angle=0, theta1=t1, theta2=t2, **kw))


def draw_pitch_heatmap(ax, xs, ys, cs=None, bins=(18, 12), cmap="YlOrRd",
                        alpha=0.82, vmin=None, vmax=None, bg="#1a472a",
                        normalize=False, sigma=1.2):
    weights = cs if cs is not None else np.ones(len(xs))
    H, _, _ = np.histogram2d(xs, ys, bins=bins, weights=weights,
                              range=[[0, 100], [0, 100]])
    if normalize:
        t = H.sum()
        if t > 0:
            H = H / t
    H_smooth = gaussian_filter(H.T, sigma=sigma)
    draw_field(ax, bg=bg)
    ax.imshow(H_smooth, cmap=cmap, alpha=alpha, origin="lower",
              extent=[0, 100, 0, 100], aspect="auto",
              vmin=vmin or 0, interpolation="bilinear", zorder=2)


# ── Dados ────────────────────────────────────────────────────────────────────
ZONE_KEYS = [
    ("def", "dir"), ("def", "ctr"), ("def", "esq"),
    ("mid", "dir"), ("mid", "ctr"), ("mid", "esq"),
    ("atk", "dir"), ("atk", "ctr"), ("atk", "esq"),
]

def classify(x, y):
    zx = "def" if x < 33 else ("mid" if x < 67 else "atk")
    zy = "dir" if y < 30 else ("ctr" if y <= 70 else "esq")
    return zx, zy

def load_points(path):
    data = json.loads(Path(path).read_text())
    pts = data.get("points", data) if isinstance(data, dict) else data
    xs, ys, cs = [], [], []
    for p in pts:
        x, y, c = p["x"], p["y"], p.get("count", 1)
        if x >= 98 and y <= 1:
            continue
        xs.append(x); ys.append(y); cs.append(c)
    return np.array(xs), np.array(ys), np.array(cs)

def build_vector(path):
    xs, ys, cs = load_points(path)
    counts = {k: 0 for k in ZONE_KEYS}
    total = 0
    for x, y, c in zip(xs, ys, cs):
        counts[classify(x, y)] = counts.get(classify(x, y), 0) + c
        total += c
    return np.array([counts[k] / total for k in ZONE_KEYS])

heatmap_files = {
    "Raphinha (La Liga)": "../data/raw/heatmap_raphinha.json",
    "Raphinha (Copa)":    "../data/raw/heatmap_raphinha_copa.json",
    "Luiz Henrique":      "../data/raw/heatmap_luiz_henrique.json",
    "Rayan":              "../data/raw/heatmap_rayan.json",
    "Endrick":            "../data/raw/heatmap_endrick.json",
}

vectors = {name: build_vector(path) for name, path in heatmap_files.items()}
print("Carregado:", list(vectors.keys()))


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(34, 5.5))
fig.patch.set_facecolor("#0d1117")
plt.subplots_adjust(top=0.72, hspace=0, wspace=0.08)

for ax, (name, path) in zip(axes, heatmap_files.items()):
    xs, ys, cs = load_points(path)
    draw_pitch_heatmap(ax, xs, ys, cs, cmap="YlOrRd", normalize=True, sigma=0.8)

    # foto acima, titulo abaixo da foto
    add_player_img(ax, HEATMAP_PHOTO[name], zoom=0.22, xy=(0.5, 1.28))
    short = name.replace(" (La Liga)", "\n(La Liga)").replace(" (Copa)", "\n(Copa)")
    ax.set_title(short, fontsize=9, fontweight="bold",
                 color=PLAYER_COLORS[name], pad=6)

fig.text(0.5, 0.97, "Distribuicao de Presenca no Campo (SofaScore, 2025/26)",
         ha="center", fontsize=13, fontweight="bold", color="white")
plt.show()


### 5.1 Raphinha: Copa vs La Liga

Ancelotti esta usando o Raphinha de forma diferente do Flick no Barca.
Abaixo: os dois campinhos lado a lado + a diferenca entre os perfis.

In [ ]:
BINS = (22, 14)

xs_l, ys_l, cs_l = load_points("../data/raw/heatmap_raphinha.json")
xs_c, ys_c, cs_c = load_points("../data/raw/heatmap_raphinha_copa.json")

def hist_norm(xs, ys, cs, bins):
    H, xe, ye = np.histogram2d(xs, ys, bins=bins, weights=cs, range=[[0,100],[0,100]])
    t = H.sum()
    return (H / t if t > 0 else H), xe, ye

H_l, xe, ye = hist_norm(xs_l, ys_l, cs_l, BINS)
H_c, _,  _  = hist_norm(xs_c, ys_c, cs_c, BINS)

fig, axes = plt.subplots(1, 3, figsize=(21, 6))
fig.patch.set_facecolor("#0d1117")

# La Liga
draw_pitch_heatmap(axes[0], xs_l, ys_l, cs_l, bins=BINS, cmap="YlOrRd", normalize=True, sigma=0.8)
axes[0].set_title("Raphinha - La Liga\n(22 jogos, Flick/Barca)",
                  fontsize=10, fontweight="bold", color=COLORS["Raphinha"], pad=8)

# Copa
draw_pitch_heatmap(axes[1], xs_c, ys_c, cs_c, bins=BINS, cmap="YlOrRd", normalize=True, sigma=0.8)
axes[1].set_title("Raphinha - Copa\n(2 jogos, Ancelotti/Brasil)",
                  fontsize=10, fontweight="bold", color="#5ba4cf", pad=8)

# Delta suavizado
H_delta = gaussian_filter(H_c, sigma=0.8) - gaussian_filter(H_l, sigma=0.8)
lim = np.abs(H_delta).max() or 0.03
draw_field(axes[2])
axes[2].imshow(H_delta, cmap="RdYlGn", alpha=0.85, origin="lower",
               extent=[0, 100, 0, 100], aspect="auto",
               vmin=-lim, vmax=lim, interpolation="bilinear", zorder=2)
axes[2].set_title("Delta: Copa - La Liga\n(verde=mais Copa, vermelho=mais LaLiga)",
                  fontsize=10, fontweight="bold", color="white", pad=8)

liga_vec = vectors["Raphinha (La Liga)"]
copa_vec = vectors["Raphinha (Copa)"]
cos_sim = float(np.dot(liga_vec, copa_vec) / (np.linalg.norm(liga_vec) * np.linalg.norm(copa_vec)))
plt.suptitle(f"Raphinha: Copa vs La Liga  |  Similaridade coseno: {cos_sim:.2f}",
             fontsize=13, fontweight="bold", color="white", y=1.02)
plt.tight_layout(pad=1.5)
plt.show()

print(
    "Leitura:\n"
    "  - Na Copa, Raphinha esta MUITO mais concentrado no Flanco Direito (seu lado natural).\n"
    "  - Na La Liga com Flick, ele era bilateral - circulava por toda a largura do campo.\n"
    "  - Ancelotti esta usando Raphinha como ponta mais fixo, com menos liberdade de inversao.\n"
    "  - Similaridade coseno 0.77 = perfis distintos para o mesmo jogador."
)


## 6. Ajuste Tatico — Dois Cenarios

**Cenario A:** similaridade vs Raphinha La Liga (22 jogos, Flick/Barca) — quem joga como o Raphinha bilateral?

**Cenario B:** similaridade vs Raphinha Copa (2 jogos, Ancelotti/Brasil) — quem joga como Ancelotti esta usando?

> Caveat Cenario B: amostra pequena (84 pontos vs 939 na La Liga). O padrao pode mudar conforme o Brasil avanca.

In [ ]:
ref_a = vectors["Raphinha (La Liga)"]
ref_b = vectors["Raphinha (Copa)"]

sim_a, sim_b = {}, {}
for name in CANDIDATES:
    v = vectors[name]
    sim_a[name] = round(float(np.dot(ref_a, v) / (np.linalg.norm(ref_a) * np.linalg.norm(v))) * 10, 2)
    sim_b[name] = round(float(np.dot(ref_b, v) / (np.linalg.norm(ref_b) * np.linalg.norm(v))) * 10, 2)

print("Cenario A - vs Raphinha La Liga (Flick):")
for nome, score in sorted(sim_a.items(), key=lambda x: -x[1]):
    print(f"  {nome:20} {score:.2f}/10")

print("\nCenario B - vs Raphinha Copa (Ancelotti):")
for nome, score in sorted(sim_b.items(), key=lambda x: -x[1]):
    print(f"  {nome:20} {score:.2f}/10")

print(
    "\nInterpretacao:"
    "\n  Cenario A: Luiz Henrique lidera por ser bilateral, como o Raphinha do Barca."
    "\n  Cenario B: Rayan e Endrick lideram por ficarem fixos no corredor, como Ancelotti quer."
    "\n  O ranking inverteu dependendo do cenario de referencia."
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (sim_dict, titulo, subtitulo) in zip(axes, [
    (sim_a, "Cenario A", "vs Raphinha La Liga (Flick)"),
    (sim_b, "Cenario B", "vs Raphinha Copa (Ancelotti)"),
]):
    nomes  = list(sim_dict.keys())
    scores = list(sim_dict.values())
    cores  = [COLORS[n] for n in nomes]
    bars   = ax.barh(nomes, scores, color=cores, alpha=0.85, edgecolor="white")
    ax.set_xlim(0, 10.5)
    ax.axvline(7, color="gray", linestyle=":", alpha=0.5)
    for bar, val in zip(bars, scores):
        ax.text(val + 0.1, bar.get_y() + bar.get_height()/2, f"{val:.2f}",
                va="center", fontweight="bold")
    ax.set_title(f"{titulo}\n{subtitulo}", fontweight="bold")
    ax.set_xlabel("Similaridade coseno (0-10)")

    # fotos no lugar dos labels do eixo y
    ax.tick_params(axis="y", labelleft=False, length=0)
    for i, player in enumerate(nomes):
        img = load_img(PLAYER_PHOTO[player])
        if img is not None:
            im = OffsetImage(img, zoom=0.20)
            ab = AnnotationBbox(im, (-0.04, i),
                               xycoords=("axes fraction", "data"),
                               box_alignment=(1.0, 0.5),
                               frameon=False, clip_on=False, zorder=10)
            ax.add_artist(ab)

plt.suptitle("Ajuste Tatico por Cenario - Similaridade Posicional com Raphinha",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.subplots_adjust(left=0.10)
plt.show()


## 7. Modelo de Decisao Multicriterio (MCDA)

Normalizacao Min-Max sobre os 3 candidatos (Raphinha excluido - e o benchmark).
Rodado nos dois cenarios para ver se a recomendacao muda.

### Criterios e Pesos
| Criterio | Peso | Fonte | Racional |
|---|---|---|---|
| Ajuste Tatico (heatmap) | 30% | Similaridade coseno vs Raphinha | Fit posicional e o mais critico para substituicao direta |
| Gols / 90 min | 20% | SofaScore | Efetividade de finalizacao isolada — diferencia perfis de 9 vs criador |
| G+A / 90 min | 10% | SofaScore | Output ofensivo agregado (complementa goals_90) |
| Experiencia Selecao | 20% | Qualitativo (historico Ancelotti + caps) | Adaptacao ao estilo da Selecao |
| Rating Medio | 15% | SofaScore | Consistencia geral na temporada |
| Minutos na Temporada | 5% | SofaScore | Volume de jogo |

> **goals/90 entra separado de G+A/90** para que a efetividade de finalizacao (Endrick: 0.37/90)
> seja distinguida da producao criativa (Rayan: maior contribuicao em assistencias).


In [ ]:
def run_mcda(df_cand, ajuste_col, label):
    criterios = {
        ajuste_col:            0.30,
        "goals_90":            0.20,
        "g_a_90":              0.10,
        "experiencia_selecao": 0.20,
        "rating_avg":          0.15,
        "minutes":             0.05,
    }
    df_s = df_cand[["player"]].copy()
    for col, peso in criterios.items():
        mn, mx = df_cand[col].min(), df_cand[col].max()
        df_s[col] = ((df_cand[col] - mn) / (mx - mn)) * 100 if mx > mn else 50
        df_s[f"{col}_pond"] = df_s[col] * peso
    ponds = [c for c in df_s.columns if "_pond" in c]
    df_s["score"] = df_s[ponds].sum(axis=1)
    return df_s[["player"] + list(criterios.keys()) + ["score"]].sort_values("score", ascending=False).round(2)

# Adicionar scores de ajuste nos dois cenarios
df["sim_a"] = df["player"].map(sim_a)
df["sim_b"] = df["player"].map(sim_b)

ranking_a = run_mcda(df, "sim_a", "Cenario A")
ranking_b = run_mcda(df, "sim_b", "Cenario B")

print("=== CENARIO A (referencia La Liga / Flick) ===")
print(ranking_a.to_string(index=False))
print()
print("=== CENARIO B (referencia Copa / Ancelotti) ===")
print(ranking_b.to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (ranking, titulo) in zip(axes, [
    (ranking_a, "Cenario A - Raphinha La Liga (Flick)"),
    (ranking_b, "Cenario B - Raphinha Copa (Ancelotti)"),
]):
    cores = [COLORS[p] for p in ranking["player"]]
    bars = ax.barh(ranking["player"], ranking["score"], color=cores, alpha=0.85, edgecolor="white")
    ax.set_xlim(0, 100)
    for bar, val in zip(bars, ranking["score"]):
        ax.text(val + 1, bar.get_y() + bar.get_height()/2, f"{val:.1f}",
                va="center", fontweight="bold")
    ax.set_title(titulo, fontweight="bold")
    ax.set_xlabel("Pontuacao Final (0-100)")

    ax.tick_params(axis="y", labelleft=False, length=0)
    for i, player in enumerate(ranking["player"]):
        img = load_img(PLAYER_PHOTO[player])
        if img is not None:
            im = OffsetImage(img, zoom=0.20)
            ab = AnnotationBbox(im, (-0.04, i),
                               xycoords=("axes fraction", "data"),
                               box_alignment=(1.0, 0.5),
                               frameon=False, clip_on=False, zorder=10)
            ax.add_artist(ab)

plt.suptitle("Ranking MCDA por Cenario", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.subplots_adjust(left=0.10)
plt.show()


## 8. Radar Chart - Perfil dos Candidatos

In [ ]:
categorias_radar = ["goals_90", "assists_90", "g_a_90", "shots_on_90", "rating_avg",
                    "sim_a", "sim_b", "experiencia_selecao"]
labels_radar = ["Gols/90", "Ass/90", "G+A/90", "Fin.Gol/90", "Rating",
                "Ajuste (A)", "Ajuste (B)", "Exp.Selecao"]

df_r = df[["player"] + categorias_radar].copy()
for cat in categorias_radar:
    mn, mx = df_r[cat].min(), df_r[cat].max()
    df_r[cat] = ((df_r[cat] - mn) / (mx - mn)) * 10 if mx > mn else 5

N = len(categorias_radar)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))

for _, row in df_r.iterrows():
    vals = row[categorias_radar].values.tolist() + [row[categorias_radar[0]]]
    ax.plot(angles, vals, "o-", linewidth=2, color=COLORS[row["player"]])
    ax.fill(angles, vals, alpha=0.10, color=COLORS[row["player"]])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels_radar, size=10)
ax.set_ylim(0, 10)
ax.set_title("Perfil Comparativo dos Candidatos (normalizado 0-10)", fontweight="bold", pad=20)

# legenda com fotos no lugar do texto
for i, row in enumerate(df_r.itertuples()):
    player = row.player
    y_pos  = 1.10 - i * 0.13
    img = load_img(PLAYER_PHOTO[player])
    if img is not None:
        im = OffsetImage(img, zoom=0.20)
        ab = AnnotationBbox(im, (1.25, y_pos), xycoords="axes fraction",
                           frameon=True,
                           bboxprops=dict(edgecolor=COLORS[player], lw=2, boxstyle="round,pad=0.1"),
                           zorder=10)
        ax.add_artist(ab)
    ax.text(1.40, y_pos, player, transform=ax.transAxes,
            va="center", ha="left", color=COLORS[player],
            fontsize=10, fontweight="bold")

plt.tight_layout()
plt.show()


## 8. Analise de Sensibilidade

Variando o peso do Ajuste Tatico de 0% a 50%, checamos se a recomendacao e robusta.
Rodado nos dois cenarios.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, (sim_col, titulo) in zip(axes, [
    ("sim_a", "Cenario A (La Liga)"),
    ("sim_b", "Cenario B (Copa)"),
]):
    resultados = []
    outros_criterios = {"goals_90": 0.20, "g_a_90": 0.10, "experiencia_selecao": 0.20, "rating_avg": 0.15, "minutes": 0.05}
    total_outros = sum(outros_criterios.values())

    for peso_at in np.arange(0, 0.55, 0.05):
        fator = (1 - peso_at) / total_outros
        crit = {sim_col: peso_at, **{k: v * fator for k, v in outros_criterios.items()}}
        for _, row in df.iterrows():
            score = 0
            for col, peso in crit.items():
                mn, mx = df[col].min(), df[col].max()
                norm = ((row[col] - mn) / (mx - mn)) * 100 if mx > mn else 50
                score += norm * peso
            resultados.append({"player": row["player"], "peso": peso_at, "score": score})

    df_s = pd.DataFrame(resultados)
    for jogador in CANDIDATES:
        sub = df_s[df_s["player"] == jogador].sort_values("peso")
        ax.plot(sub["peso"] * 100, sub["score"], marker="o", linewidth=2,
                color=COLORS[jogador])
        # foto no final da linha (peso=50%)
        last = sub.iloc[-1]
        img = load_img(PLAYER_PHOTO[jogador])
        if img is not None:
            im = OffsetImage(img, zoom=0.16)
            ab = AnnotationBbox(im, (last["peso"] * 100, last["score"]),
                               frameon=True,
                               bboxprops=dict(edgecolor=COLORS[jogador], lw=1.5,
                                              boxstyle="round,pad=0.1"),
                               zorder=10)
            ax.add_artist(ab)

    ax.set_title(titulo, fontweight="bold")
    ax.set_xlabel("Peso do Ajuste Tatico (%)")
    ax.set_ylabel("Pontuacao Final")
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 100)

plt.suptitle("Sensibilidade ao Peso do Ajuste Tatico - por Cenario", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
w_a = ranking_a[["player","score"]].set_index("player")["score"].to_dict()
w_b = ranking_b[["player","score"]].set_index("player")["score"].to_dict()
venc_a = ranking_a.iloc[0]["player"]
venc_b = ranking_b.iloc[0]["player"]
print(f"Cenario A: {venc_a} ({w_a[venc_a]:.1f}) | Cenario B: {venc_b} ({w_b[venc_b]:.1f})")


## 9. Conclusao

### O que os dados mostram

**Cenario A — Quem substitui o Raphinha do Barca (Flick)?**
**Luiz Henrique** (65.9 pts). O perfil bilateral, com cobertura ampla e sem flanco dominante,
e o mais similar ao Raphinha que circulava livremente no sistema do Flick (similaridade coseno = 8.96).

**Cenario B — Quem substitui o Raphinha que Ancelotti esta usando na Copa?**
**Endrick** (79.0 pts), com folga sobre Rayan (53.4 pts).
Com goals_90 como criterio independente (20%), a efetividade de finalizacao do Endrick
diferencia os dois candidatos que tinham fit posicional quase identico (9.73 vs 9.70).
Ancelotti esta usando Raphinha fixo no corredor direito — Endrick combina esse fit com
maior perigo ao gol, o que vale mais num torneio eliminatorio.

---

### Recomendacao

| Cenario | Opcao | Score | Justificativa |
|---|---|---|---|
| **B (Copa)** | **Endrick** | 79.0 | Melhor fit no corredor direito (Ancelotti) + maior efetividade de finalizacao (0.37 gols/90 + 0.52 ass/90). |
| **A (La Liga)** | **Luiz Henrique** | 65.9 | Perfil bilateral mais proximo do Raphinha do Flick. Maior experiencia com Ancelotti na Selecao. |
| Alternativa | Rayan | 53.4 (B) | Ponta classica de corredor, fit posicional alto, mas sem experiencia internacional e rating inferior. |

---

### Por que o empate entre Endrick e Rayan foi resolvido

Antes (apenas G+A/90 agregado): Rayan=9.73 vs Endrick=9.70 — quase identicos em ajuste tatico.
Apos separar goals_90 (20%) de G+A/90 (10%):
- Endrick: goals_90=0.37, assists_90=0.52 → produtividade ofensiva real e alta
- Rayan: goals_90=0.40, rating baixo, sem experiencia selecao → penalizado nos demais criterios

---

### Limitacoes
- Copa do Mundo: apenas 2 jogos para o Raphinha (amostra pequena para o Cenario B).
- xG indisponivel para Luiz Henrique (RFPL sem parceria com Opta/StatsBomb).
- `rating_avg` e `experiencia_selecao` ainda tem componente qualitativo.

**Fonte:** SofaScore - coleta manual via extensao Chrome em 22/06/2026.
